In [63]:
import re
import os
import sys
import pandas as pd

In [64]:
# This script is intended to re-structure the strain names from what I usually use (fine for Beast1) to what Beast2 needs

In [65]:
# First, read in the documents to be re-named

In [66]:
#Function: Read in the FASTA as a dataframe

def fasta_reader(path_to_fasta, output_name):
    fasta_data = []
    FASTANAME = path_to_fasta
        
    with open(FASTANAME) as f:
        header = ""
        sequence = ""
        for line in f:
            if line.startswith(">"):
                if header != "":
                    fasta_data.append({"header": header, "sequence": sequence})
                header = line.strip() 
                sequence = ""
            else:
                sequence += line.strip()
        fasta_data.append({"header": header, "sequence": sequence}) #last line 
        
    globals()[output_name] = pd.DataFrame(fasta_data)

    return

In [67]:
# Read in your FASTA 

# FASTA = "../BEAST/empirical_trees/submission_files/2026-01_main-intro_equal-time-loc_ha_empirical_targetedbeast/div-tree-strains.fasta"
# FASTA = "../BEAST/empirical_trees/submission_files/2026-01_main-intro_equal-time-host-1000_ha_empirical_targetedbeast/div-tree-strains.fasta"
# FASTA = "../BEAST/empirical_trees/submission_files/2026-01_main-intro_equal-time-host-3000_ha_empirical_targetedbeast/div-tree-strains.fasta"
# FASTA = "../BEAST/empirical_trees/submission_files/2026-02-04_main-intro_prop-time-host_ha_empirical_targetedbeast/inputs/prop_time-host_strains.fasta"
# FASTA = "../BEAST/empirical_trees/submission_files/2026-02-04_main-intro_prop-time-loc_ha_empirical_targetedbeast/inputs/prop_time-loc_strains.fasta"
# FASTA = "../BEAST/empirical_trees/submission_files/2026-02-04_main-intro_liveharvest_ha_empirical_targetedbeast/inputs/liveharvest_strains.fasta"
# FASTA = "../../H5-genotypes/updated_datasets-withGenome/A1_A2/A1_A2_genome.fasta"
FASTA = "../../H5-genotypes/updated_datasets-withGenome/A3/A3_genome_d11ancestral.fasta"
# FASTA = "../../H5-genotypes/updated_datasets-withGenome/B3.2/B3.2_genome.fasta"
# FASTA = "../../H5-genotypes/updated_datasets-withGenome/B3.6/B3.6_genome.fasta"
# FASTA = "../../H5-genotypes/updated_datasets-withGenome/C2.1/C2.1_genome.fasta"
# FASTA = "../../H5-genotypes/updated_datasets-withGenome/D1.1/D1.1_genome.fasta"

fasta_reader(FASTA, "ha_alignment")

In [68]:
# Function: remove the carrot from the header

def header_to_strain(df):
    df["strain"] = df["header"].str.replace(">","")

    return(df)

In [69]:
# Remove the ">" in the headers

ha_alignment_strain = header_to_strain(ha_alignment)

In [70]:
# Read in a TSV

# FILE = "../BEAST/empirical_trees/submission_files/2026-01_main-intro_equal-time-loc_ha_empirical_targetedbeast/div-tree-metadata.tsv"
# FILE = "../BEAST/empirical_trees/submission_files/2026-01_main-intro_equal-time-host-1000_ha_empirical_targetedbeast/div-tree-metadata.tsv"
# FILE = "../BEAST/empirical_trees/submission_files/2026-01_main-intro_equal-time-host-3000_ha_empirical_targetedbeast/div-tree-metadata.tsv"
# FILE = "../BEAST/empirical_trees/submission_files/2026-02-04_main-intro_prop-time-host_ha_empirical_targetedbeast/inputs/prop_time-host_metadata.tsv"
# FILE = "../BEAST/empirical_trees/submission_files/2026-02-04_main-intro_prop-time-loc_ha_empirical_targetedbeast/inputs/prop_time-loc_metadata.tsv"
# FILE = "../BEAST/empirical_trees/submission_files/2026-02-04_main-intro_liveharvest_ha_empirical_targetedbeast/inputs/liveharvest_metadata.tsv"

# FILE1 = "../../H5-genotypes/updated_datasets-withGenome/A1_A2/A1_strains.tsv"
# FILE2 = "../../H5-genotypes/updated_datasets-withGenome/A1_A2/A2_strains.tsv"
FILE = "../../H5-genotypes/updated_datasets-withGenome/A3/A3_strains.tsv"
# FILE = "../../H5-genotypes/updated_datasets-withGenome/B3.2/B3.2_strains.tsv"
# FILE = "../../H5-genotypes/updated_datasets-withGenome/B3.6/B3.6_strains.tsv"
# FILE = "../../H5-genotypes/updated_datasets-withGenome/C2.1/C2.1_strains.tsv"
# FILE = "../../H5-genotypes/updated_datasets-withGenome/D1.1/D1.1_strains.tsv"

metadata = pd.read_csv(FILE, sep = '\t')

In [71]:
print(metadata["strain"].is_unique)

for strain, value in metadata["strain"].value_counts().items():
    if value == 8:
        pass
    else:
        print(f"strain {strain} has an incomplete genome")

metadata_unique = metadata[metadata["ncbibvbrc_segment"] == 4]

False


In [72]:
# Function: reorder the strain name so date is at the end

def reorder_strain(df, column):

    parts = df[column].str.split("|", expand = True)

    df["new-strain"] = parts[0] + "|" + parts[2] + "|" + parts[3] + "|" + parts[4] + "|" + parts[1]

    return(df)

In [73]:
ha_alignment_new_strain = reorder_strain(ha_alignment_strain, "strain")

In [74]:
metadata_new_strain = reorder_strain(metadata_unique, "strain")

/var/folders/8z/j5vh6t990rj40lqqxtsr69z00000gp/T/ipykernel_74470/3847559030.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["new-strain"] = parts[0] + "|" + parts[2] + "|" + parts[3] + "|" + parts[4] + "|" + parts[1]


In [75]:
# There is a small, slight problem with some of the FASTAs having sequences with no sequence
# If I generate an alignment using the Nextstrain augur pipeline, it filters out null sequences
# However, if I generate FASTAs by myself (which I did for proportional and live/harvest datasets), there are still null sequences

def remove_null(fasta_df, metadata_df):

    null_strains = list(fasta_df["new-strain"].loc[fasta_df["sequence"] == "None"])
    print(len(null_strains))

    fasta_df_upt = fasta_df[~fasta_df["new-strain"].isin(null_strains)]
    metadata_df_upt = metadata_df[~metadata_df["new-strain"].isin(null_strains)]

    return(fasta_df_upt, metadata_df_upt)

In [76]:
ha_alignment_new_strain_nonull, metadata_new_strain_nonull = remove_null(ha_alignment_new_strain, metadata_new_strain)

0


In [77]:
# I also want to know which strains have ambiguous dates
# For BEAST2, I have to input the nonambiguous date as the tip date
# And THEN I have to add a tip prior for those tips with ambiguous dates
# Its stupid and I hate it

def flag_ambiguous_dates(metadata_df):

    ambig = metadata_df["new-strain"].loc[metadata_df["ncbibvbrc_date"].str.contains("X", regex = False)]
    print(len(ambig))

    return(ambig)

In [78]:
ambig_strains = flag_ambiguous_dates(metadata_new_strain_nonull)

0


In [79]:
# ambig_strains.to_csv("../BEAST/empirical_trees/submission_files/2026-01_main-intro_equal-time-loc_ha_empirical_targetedbeast/eq-time-loc_ambiguous_dates.tsv", sep = "\t", index = False, header = False)
# ambig_strains.to_csv("../BEAST/empirical_trees/submission_files/2026-01_main-intro_equal-time-host-1000_ha_empirical_targetedbeast/eq-time-host-1000_ambiguous_dates.tsv", sep = "\t", index = False, header = False)
# ambig_strains.to_csv("../BEAST/empirical_trees/submission_files/2026-01_main-intro_equal-time-host-3000_ha_empirical_targetedbeast/eq-time-host-3000_ambiguous_dates.tsv", sep = "\t", index = False, header = False)
# ambig_strains.to_csv("../BEAST/empirical_trees/submission_files/2026-02-04_main-intro_prop-time-host_ha_empirical_targetedbeast/inputs/prop-time-host_ambiguous_dates.tsv", sep = "\t", index = False, header = False)
# ambig_strains.to_csv("../BEAST/empirical_trees/submission_files/2026-02-04_main-intro_prop-time-loc_ha_empirical_targetedbeast/inputs/prop-time-loc_ambiguous_dates.tsv", sep = "\t", index = False, header = False)
# ambig_strains.to_csv("../BEAST/empirical_trees/submission_files/2026-02-04_main-intro_liveharvest_ha_empirical_targetedbeast/inputs/live_ambiguous_dates.tsv", sep = "\t", index = False, header = False)

# ambig_strains.to_csv("../../H5-genotypes/A1_A2_ambiguous-dates.tsv", sep = "\t", index = False, header = False)
ambig_strains.to_csv("../../H5-genotypes/A3_ambiguous-dates.tsv", sep = "\t", index = False, header = False)
# ambig_strains.to_csv("../../H5-genotypes/B3.2_ambiguous-dates.tsv", sep = "\t", index = False, header = False)
# ambig_strains.to_csv("../../H5-genotypes/B3.6_ambiguous-dates.tsv", sep = "\t", index = False, header = False)
# ambig_strains.to_csv("../../H5-genotypes/C2.1_ambiguous-dates.tsv", sep = "\t", index = False, header = False)
# ambig_strains.to_csv("../../H5-genotypes/D1.1_ambiguous-dates.tsv", sep = "\t", index = False, header = False)

In [80]:
# Function to write a new fasta with just the sequences in the metadata
# Adapted from Maria

def fasta_writer(path, filename, df):
            
    try:  
        os.mkdir(path)

    except OSError as error:
        pass

    with open(f"{path}{filename}", "w") as f:
        for index, row in df.iterrows():
            f.write(f">{row['new-strain']}\n")
            f.write(f"{row['sequence']}\n")

In [81]:
# Print the new FASTA 

# fasta_writer("../BEAST/empirical_trees/submission_files/2026-01_main-intro_equal-time-loc_ha_empirical_targetedbeast/", "div-tree-strains-newheader.fasta", ha_alignment_new_strain)
# fasta_writer("../BEAST/empirical_trees/submission_files/2026-01_main-intro_equal-time-host-1000_ha_empirical_targetedbeast/", "div-tree-strains-newheader.fasta", ha_alignment_new_strain)
# fasta_writer("../BEAST/empirical_trees/submission_files/2026-01_main-intro_equal-time-host-3000_ha_empirical_targetedbeast/", "div-tree-strains-newheader.fasta", ha_alignment_new_strain)
# fasta_writer("../BEAST/empirical_trees/submission_files/2026-02-04_main-intro_prop-time-host_ha_empirical_targetedbeast/inputs/", "prop_time-host_strains-newheader.fasta", ha_alignment_new_strain_nonull)
# fasta_writer("../BEAST/empirical_trees/submission_files/2026-02-04_main-intro_prop-time-loc_ha_empirical_targetedbeast/inputs/", "prop_time-loc_strains-newheader.fasta", ha_alignment_new_strain_nonull)
# fasta_writer("../BEAST/empirical_trees/submission_files/2026-02-04_main-intro_liveharvest_ha_empirical_targetedbeast/inputs/", "liveharvest_strains-newheader.fasta", ha_alignment_new_strain_nonull)

# fasta_writer("../../H5-genotypes/", "A1_A2_genome_newheader.fasta", ha_alignment_new_strain_nonull)
fasta_writer("../../H5-genotypes/", "A3_genome_d11ancestral_newheader.fasta", ha_alignment_new_strain_nonull)
# fasta_writer("../../H5-genotypes/", "B3.2_genome_newheader.fasta", ha_alignment_new_strain_nonull)
# fasta_writer("../../H5-genotypes/", "B3.6_genome_newheader.fasta", ha_alignment_new_strain_nonull)
# fasta_writer("../../H5-genotypes/", "C2.1_genome_newheader.fasta", ha_alignment_new_strain_nonull)
# fasta_writer("../../H5-genotypes/", "D1.1_genome_newheader.fasta", ha_alignment_new_strain_nonull)

In [82]:
metadata_ancestral = metadata_new_strain_nonull.loc[metadata_new_strain_nonull["new-strain"].isin(ha_alignment_new_strain_nonull["new-strain"].to_list())]
print(len(metadata_ancestral))

114


In [83]:
# While you're at it, simply print the lat/longs using the new strain name if necessary

In [89]:
def subset_dataframe(metadata):

    # needed_columns = ["new-strain", "lat", "long"]
    # needed_columns = ["new-strain", "ncbibvbrc_hostgroup"]
    needed_columns = ["new-strain", "ncbibvbrc_nonambigdate"]
    metadata_subset = metadata[needed_columns]

    # metadata_subset.dropna(subset=['lat'], inplace=True)

    return(metadata_subset)

In [90]:
metadata_subset = subset_dataframe(metadata_ancestral)

In [91]:
# metadata_subset.to_csv("../BEAST/empirical_trees/submission_files/2026-01_main-intro_equal-time-loc_ha_empirical_targetedbeast/div-tree-date.tsv", sep = "\t", index = False, header = False)
# metadata_subset.to_csv("../BEAST/empirical_trees/submission_files/2026-01_main-intro_equal-time-host-1000_ha_empirical_targetedbeast/div-tree-date.tsv", sep = "\t", index = False, header = False)
# metadata_subset.to_csv("../BEAST/empirical_trees/submission_files/2026-01_main-intro_equal-time-host-3000_ha_empirical_targetedbeast/div-tree-date.tsv", sep = "\t", index = False, header = False)
# metadata_subset.to_csv("../BEAST/empirical_trees/submission_files/2026-02-04_main-intro_prop-time-host_ha_empirical_targetedbeast/inputs/prop_time-host_lat-long.tsv", sep = "\t", index = False, header = False)
# metadata_subset.to_csv("../BEAST/empirical_trees/submission_files/2026-02-04_main-intro_prop-time-loc_ha_empirical_targetedbeast/prop_time-loc_date.tsv", sep = "\t", index = False, header = False)
# metadata_subset.to_csv("../BEAST/empirical_trees/submission_files/2026-02-04_main-intro_liveharvest_ha_empirical_targetedbeast/liveharvest_lat-long.tsv", sep = "\t", index = False, header = False)

# metadata_subset.to_csv("../../H5-genotypes/A1_A2_host.txt", sep = "\t", index = False, header = False)
metadata_subset.to_csv("../../H5-genotypes/A3_date.txt", sep = "\t", index = False, header = False)
# metadata_subset.to_csv("../../H5-genotypes/B3.2_host.txt", sep = "\t", index = False, header = False)
# metadata_subset.to_csv("../../H5-genotypes/B3.6_host.txt", sep = "\t", index = False, header = False)
# metadata_subset.to_csv("../../H5-genotypes/C2.1_host.txt", sep = "\t", index = False, header = False)
# metadata_subset.to_csv("../../H5-genotypes/D1.1_host.txt", sep = "\t", index = False, header = False)